In [1]:
from utils.orthanc_utils import *
from utils.db_utils import *
from utils.plot_utils import *
from utils.pipeline import *


def get_volumes(dicoms):    
    df = pd.DataFrame(
        [{"SliceLocation": d.SliceLocation, "InstanceNumber": d.InstanceNumber, "PixelArray": d.pixel_array} for d in dicoms]
    ).sort_values(["SliceLocation", "InstanceNumber"])

    d0 = dicoms[0]
    pixel_spacing = float(d0.PixelSpacing[0])
    thickness = float(getattr(d0, "SpacingBetweenSlices", getattr(d0, "SliceThickness", np.nan)))
    voxel_size = (pixel_spacing ** 2 * thickness) / 1000
    
    array_4d = []
    for _, slice_df in df.groupby("SliceLocation"):
        time_array = []
        for _, time_df in slice_df.groupby("InstanceNumber"):
            pixel_array = time_df["PixelArray"].iloc[0]
            time_array.append(pixel_array)

        slice_array = np.stack(time_array, axis=-1)
        array_4d.append(slice_array)

    array_4d = np.stack(array_4d, axis=-2)

    labels = {500: "lv", 1000: "rv", 1500: "lv_myo", 2000: "rv_myo"}
    curves = {
        name: np.sum(array_4d == val, axis=(0, 1, 2)) * voxel_size
        for val, name in labels.items()
    }
    lv, rv = curves["lv"], curves["rv"]
    lv_edv, lv_esv = lv.max(), lv.min()
    rv_edv, rv_esv = rv.max(), rv.min()

    metrics = {
        "lv_edv": lv_edv,
        "lv_esv": lv_esv,
        "lv_ef": (lv_edv - lv_esv) / lv_edv,
        "lv_mass": np.mean(curves["lv_myo"]) * 1.05,
        "rv_edv": rv_edv,
        "rv_esv": rv_esv,
        "rv_ef": (rv_edv - rv_esv) / rv_edv,
        "rv_mass": np.mean(curves["rv_myo"]) * 1.05,
    }

    return metrics


2026-04-10 12:19:42.432 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/home/tina/Documents/PhD/Project/imageCLASP/clasp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-10 12:19:42.521 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 12:19:42.522 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-04-10 12:19:42.522 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in

In [3]:
db = TinyDB(DB_PATH)
studies = fetch_db_studies()

study = studies[0]

'cc6fb873-9cbebf9c-322a7c90-85eadd68-56912070'

In [15]:
db = TinyDB(DB_PATH)
studies = fetch_db_studies()

study_dict = {}
for study in studies:
    df = pd.DataFrame([series.__dict__ for series in study.series_dict.values()])
    sax_dl_df = df[(df['dl_orthanc_id'].notna()) & (df['roundel_orthanc_id'].isna())]
    if len(sax_dl_df) > 0:
        study_dict[f'{study.patient_id} | {study.patient_name} | {study.study_date}'] = study

In [18]:
study

In [17]:
sax_dl_df

,orthanc_series_id,series_uid,series_description,series_type,series_group,series_orientation,dl_orthanc_id,roundel_orthanc_id
5,07618786-41b29ba5-74969b9e-0ecda847-2c4f2495,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,9d2aae58-63b5dbc4-683b7b3f-971e03ef-29f13703,None
7,0870d370-24d65155-dbfb45c9-cddf209a-c196ecaf,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,27567c90-947d46fb-8cbb6204-ca1bebd6-9d593c61,None
31,3db48026-9e0d1e3f-2cb756fc-2fb247f5-7bcf5019,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,4d1ae2ac-74af2a8a-67fc7a1b-813a3aae-60e01bd1,None
36,41721aee-d759497a-bdaa1948-7322764d-8c19a0eb,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,e7f5e1a8-b8e50c94-7b1c823e-44cfd6a2-ffcdacaa,None
58,5f76438c-9e082466-bae705d6-4cb1b465-cc8a27ca,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,e7b02609-c0047886-7699b978-9ea9a879-033e3415,None
65,710d7088-ee22db64-bd9feebf-bc55af58-d30e75df,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,d23bd8b4-cfec2299-041f8d8b-b6464ce4-84aa314b,None
69,75b4763c-79b4fab2-bd4e6315-38e793d8-f1a34d01,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,0430543a-3870d455-4ff3281d-593de98b-17d806be,None
84,93f4ac29-37a1e29c-5827e22c-db04e454-9b48f05c,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,e960e708-a0cb40d1-2fd351b9-8e3a2219-c39cc43f,None
87,97ecf97c-5b9fed67-18c1185a-defa10b3-06383564,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,e7a933ce-3ea89d37-d28a8d38-41e943bf-57344207,None
92,9ea970b7-8b2953ca-bc0d05d7-a5b20be0-e14a6fe3,1.3.12.2.1107.5.2.18.42363.2025123109282085086...,SSFP,Cine Stack,Cine Stack:00,SAX,f7523730-4e15087b-c9006230-d0db47db-9c1ec6e1,None


In [16]:
study.__dict__

{'series_dict': {'017c4324-8b4349c0-5f8698dd-d71527b5-66304ee6': <utils.db_utils.Series at 0x79cb48b79970>,
  '03f9e3fb-adadeb87-ddb96ec5-2b0a9e51-c8c7a224': <utils.db_utils.Series at 0x79cb48bbbec0>,
  '04591445-a4091023-19874f79-94b6e33f-1ce26f4c': <utils.db_utils.Series at 0x79cb48bbb620>,
  '05359c0c-fd5e1e62-f4799a13-0ee097e5-de442f95': <utils.db_utils.Series at 0x79cb48bb91c0>,
  '0726b782-d035d87d-c90bb21c-adc4def7-9ab582b4': <utils.db_utils.Series at 0x79cb48bbb080>,
  '07618786-41b29ba5-74969b9e-0ecda847-2c4f2495': <utils.db_utils.Series at 0x79cb48bbbe30>,
  '0817a808-21fd44a8-72c38420-08a2df93-9f3ab962': <utils.db_utils.Series at 0x79cb48bbb680>,
  '0870d370-24d65155-dbfb45c9-cddf209a-c196ecaf': <utils.db_utils.Series at 0x79cb48bbb830>,
  '0b15c697-dcf915a8-a4afe4e1-c4aa6b2d-8638afa0': <utils.db_utils.Series at 0x79cb48bbb020>,
  '0ce4c012-242315ad-ad505759-0813e844-25af8931': <utils.db_utils.Series at 0x79cb48bba900>,
  '0d4de95f-1d118949-cf0e00fd-cae4e9fe-fb5b2e00': <util

In [10]:
study_list

{}

In [3]:
df = pd.DataFrame([series.__dict__ for series in study.series_dict.values()])
sax_dl_df = df[(df['dl_orthanc_id'].notna()) & (df['roundel_orthanc_id'].isna())]

for series_id, dl_series_id in zip(sax_dl_df["orthanc_series_id"], sax_dl_df["dl_orthanc_id"]):
    series = study.series_dict[series_id]
    image_dicoms = fetch_orthanc_dicoms_for_series(series_id)
    mask_dicoms = fetch_orthanc_dicoms_for_series(dl_series_id)

    masked_images = [image.pixel_array * (mask.pixel_array > 0) for image, mask in zip(image_dicoms, mask_dicoms) ]
    new_orthanc_id = send_series_to_orthanc(masked_images, image_dicoms, new_description='Roundel Processed')
    series.roundel_orthanc_id = new_orthanc_id

metrics = get_volumes(fetch_orthanc_dicoms_for_series_list(df["dl_orthanc_id"].dropna().unique()))
metrics['orthanc_study_id'] = study.orthanc_study_id
metrics['patient_id'] = study.patient_id
metrics['study_date'] = study.study_date

metrics_df = pd.read_csv(METRICS_PATH)
metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics])]).set_index('orthanc_study_id')
metrics_df.to_csv(METRICS_PATH)


/tmp/ipykernel_203371/540095361.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics])]).set_index('orthanc_study_id')


In [4]:
metrics_df

,patient_id,study_date,lv_edv,lv_esv,lv_ef,lv_mass,rv_edv,rv_esv,rv_ef,rv_mass
orthanc_study_id,,,,,,,,,,
c84ff965-9b667fef-54070af3-b1386f1a-42841c76,67812345,20251201,30.866044,13.072321,0.576482,23.216613,58.395765,32.574646,0.442175,11.112382


In [2]:
import tk